# CineIQ — Collaborative Filtering with SVD

Collaborative filtering recommends items based on the preferences of similar users.
We use **Singular Value Decomposition (SVD)** via the Surprise library to factorize the user-item rating matrix.

## Parameters
- **n_factors = 100**: Latent factor dimensionality
- **n_epochs = 20**: Training iterations
- **Random state = 42**: Reproducibility

In [1]:
# Imports
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate
import pandas as pd
import pickle

# Load ratings
ratings = pd.read_csv('../data/processed/merged.csv')
ratings = ratings[['userId', 'movieId', 'rating']]
print(ratings.shape)

(1000209, 3)


## Training on Full Dataset
The Surprise library requires a special Dataset format. We build a full trainset and fit the SVD model on all available ratings.

In [2]:
# Surprise format
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings, reader)

# Train SVD
trainset = data.build_full_trainset()
svd = SVD(n_factors=100, n_epochs=20, random_state=42)
svd.fit(trainset)
print("SVD trained.")

SVD trained.


## 3-Fold Cross-Validation
We evaluate the SVD model using 3-fold cross-validation. The **RMSE (Root Mean Square Error)** measures prediction accuracy — lower is better.

**Result: RMSE = 0.8861** — This is a solid baseline for the MovieLens 1M dataset (typical range: 0.85–0.90).

In [3]:
# Quick cross-validation
results = cross_validate(svd, data, measures=['RMSE'], cv=3, verbose=True)

Evaluating RMSE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    0.8853  0.8876  0.8837  0.8855  0.0016  
Fit time          8.06    7.00    7.56    7.54    0.43    
Test time         4.13    2.48    2.63    3.08    0.74    


## Recommendation Function
`get_svd_recommendations(user_id, n=10)` predicts ratings for all movies the user hasn't rated yet, ranks them by predicted rating, and returns the top-n.

In [4]:
# Top N recommendations for a user
def get_svd_recommendations(user_id, n=10):
    all_movie_ids = ratings['movieId'].unique()
    rated = ratings[ratings['userId'] == user_id]['movieId'].tolist()
    unrated = [m for m in all_movie_ids if m not in rated]
    
    predictions = [(m, svd.predict(user_id, m).est) for m in unrated]
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_ids = [p[0] for p in predictions[:n]]
    
    movies = pd.read_csv('../data/processed/movies.csv')
    return movies[movies['movieId'].isin(top_ids)][['title', 'genres']]

## Testing & Saving
Testing with user_id=1 returns highly-rated classics (The Usual Suspects, Shawshank Redemption). We save the trained SVD model for use in the ensemble.

In [5]:
# Test
print(get_svd_recommendations(user_id=1))

# Save
pickle.dump(svd, open('../models/svd_model.pkl', 'wb'))
print("Saved.")

                                 title              genres
315   Shawshank Redemption, The (1994)               Drama
662             Pather Panchali (1955)               Drama
847              Godfather, The (1972)  Action|Crime|Drama
900                  Casablanca (1942)   Drama|Romance|War
941       It's a Wonderful Life (1946)               Drama
1180    Raiders of the Lost Ark (1981)    Action|Adventure
1227              Graduate, The (1967)       Drama|Romance
1880     Man for All Seasons, A (1966)               Drama
2836                    Sanjuro (1962)    Action|Adventure
2953               General, The (1927)              Comedy
Saved.
